In [1]:
import random

# -------------------
# Fitness Function
# -------------------
def fitness(ind):
    n = len(ind)
    non_attacking = n*(n-1)//2  # max possible pairs
    attacking = 0
    for i in range(n):
        for j in range(i+1, n):
            if abs(ind[i]-ind[j]) == abs(i-j):  # same diagonal
                attacking += 1
    return non_attacking - attacking

# -------------------
# Tournament Selection
# -------------------
def tournament_selection(pop, k=3):
    selected = random.sample(pop, k)
    return max(selected, key=fitness)

# -------------------
# Cycle Crossover
# -------------------
def cycle_crossover(p1,p2):
    size = len(p1)
    def create_child(parent1,parent2):
        child = [None]*size
        cycles_done = [False]*size
        index = 0
        while not all(cycles_done):
            while cycles_done[index]:
                index += 1
            start = index
            while True:
                child[index] = parent1[index]
                cycles_done[index] = True
                val = parent2[index]
                index = parent1.index(val)
                if index == start:
                    break
        for i in range(size):
            if child[i] is None:
                child[i] = parent2[i]
        return child
    return create_child(p1,p2), create_child(p2,p1)

# -------------------
# Inversion Mutation
# -------------------
def inversion_mutation(ind, rate=0.1):
    if random.random() < rate:
        i,j = sorted(random.sample(range(len(ind)),2))
        ind[i:j+1] = reversed(ind[i:j+1])
    return ind

# -------------------
# Initialize Population
# -------------------
def init_population(size, N):
    population = []
    base = list(range(N))
    for _ in range(size):
        ind = base[:]
        random.shuffle(ind)
        population.append(ind)
    return population

# -------------------
# Genetic Algorithm
# -------------------
def genetic_algo(N, pop_size=50, generations=200, mutation_rate=0.1):
    population = init_population(pop_size, N)
    best = max(population, key=fitness)
    best_fit = fitness(best)

    for gen in range(generations):
        new_pop = []
        while len(new_pop) < pop_size:
            p1 = tournament_selection(population)
            p2 = tournament_selection(population)
            c1,c2 = cycle_crossover(p1,p2)
            c1 = inversion_mutation(c1, mutation_rate)
            c2 = inversion_mutation(c2, mutation_rate)
            new_pop.extend([c1,c2])
        population = new_pop[:pop_size]

        current_best = max(population, key=fitness)
        if fitness(current_best) > best_fit:
            best_fit = fitness(current_best)
            best = current_best

        if best_fit == N*(N-1)//2:
            # solution found
            break

    return best, best_fit

# -------------------
# Solve N-Queens
# -------------------
N = 8
solution, solution_fitness = genetic_algo(N, pop_size=100, generations=500, mutation_rate=0.2)
print(f"N={N} Solution:", solution)
print("Fitness:", solution_fitness)

N=8 Solution: [6, 0, 2, 7, 5, 3, 1, 4]
Fitness: 28
